In [1]:
import warnings
warnings.filterwarnings("ignore")

# Exercițiul 1 — Problema Monty Hall (pgmpy)

## Enunțul problemei

Într-un concurs de televiziune există trei uși. În spatele uneia se află un premiu (o mașină), iar în spatele celorlalte două câte o capră. Concurentul alege o ușă. Gazda, care știe unde se află premiul, deschide **o altă ușă** în spatele căreia este o capră și îl întreabă pe concurent dacă vrea să-și schimbe alegerea cu cealaltă ușă rămasă închisă.

Întrebarea este dacă merită să schimbe ușa. Intuiția spune că cele două uși rămase au șanse egale ($1/2$ fiecare), dar rezultatul corect, obținut cu teorema lui Bayes, este că **schimbarea dublează șansele de câștig** (de la $1/3$ la $2/3$).

## Modelul probabilist

Se folosesc trei variabile aleatoare, fiecare cu trei stări (ușile $0$, $1$, $2$):

- $C$ — ușa aleasă de concurent (*Contestant*);
- $P$ — ușa în spatele căreia se află premiul (*Prize*);
- $H$ — ușa deschisă de gazdă (*Host*).

Variabilele $C$ și $P$ sunt independente și uniform distribuite: $P(C=i)=P(P=i)=\frac13$.

Gazda respectă regulile jocului: nu deschide ușa aleasă de concurent și nu deschide ușa cu premiul. Dacă $C=P$, gazda poate alege oricare dintre celelalte două uși, deci le alege cu probabilitate $\frac12$ fiecare. Dacă $C\neq P$, gazda are o singură ușă posibilă, deci o alege cu probabilitate $1$. Prin urmare $H$ depinde de $C$ și de $P$.

Rețeaua bayesiană are structura $C \rightarrow H \leftarrow P$, iar distribuția comună se factorizează astfel:

$$P(C,P,H)=P(C)\,P(P)\,P(H\mid C,P)$$

Structura $C \rightarrow H \leftarrow P$ (colizor) explică paradoxul: $C$ și $P$ sunt independente a priori, dar devin dependente atunci când $H$ este observată. De aceea, faptul că gazda a deschis o ușă anume aduce informație despre locul premiului.

## Explicarea codului

**Importurile.** `DiscreteBayesianNetwork` este clasa care reprezintă o rețea bayesiană cu variabile discrete (structura grafului), iar `TabularCPD` definește un tabel de probabilități condiționate (CPD) pentru o variabilă.

In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

model = DiscreteBayesianNetwork([("C", "H"), ("P", "H")])

cpd_c = TabularCPD("C", 3, [[0.33], [0.33], [0.33]])
cpd_p = TabularCPD("P", 3, [[0.33], [0.33], [0.33]])
cpd_h = TabularCPD(
    "H",
    3,
    [
        [0, 0, 0, 0, 0.5, 1, 0, 1, 0.5],
        [0.5, 0, 1, 0, 0, 0, 1, 0, 0.5],
        [0.5, 1, 0, 1, 0.5, 0, 0, 0, 0],
    ],
    evidence=["C", "P"],
    evidence_card=[3, 3],
)

model.add_cpds(cpd_c, cpd_p, cpd_h)

model.get_cpds()

[<TabularCPD representing P(C:3) at 0x7f1c2b9746e0>,
 <TabularCPD representing P(P:3) at 0x7f1c2bafae90>,
 <TabularCPD representing P(H:3 | C:3, P:3) at 0x7f1c2bafb110>]

**Structura rețelei.** `DiscreteBayesianNetwork([("C", "H"), ("P", "H")])` creează un graf orientat aciclic cu muchiile $C\to H$ și $P\to H$, adică $C$ și $P$ sunt părinții lui $H$.

**CPD-urile pentru $C$ și $P$.** În `TabularCPD("C", 3, [[0.33], [0.33], [0.33]])` primul argument este numele variabilei, al doilea numărul de stări (3 uși), iar al treilea tabelul de valori. Variabilele $C$ și $P$ nu au părinți, deci tabelul are o singură coloană cu cele trei probabilități. Valoarea $0.33$ aproximează $\frac13$; suma $0.99$ este acceptată de pgmpy la verificare, iar la inferență distribuția rezultată este normalizată.

**CPD-ul pentru $H$.** Tabelul are 3 rânduri (valorile $H=0,1,2$) și $3\cdot 3=9$ coloane, câte una pentru fiecare combinație de valori ale părinților. Parametrul `evidence=["C", "P"]` precizează părinții, iar `evidence_card=[3, 3]` numărul de stări ale fiecăruia. Coloanele sunt ordonate astfel încât ultimul părinte ($P$) variază cel mai repede:

| | $C0,P0$ | $C0,P1$ | $C0,P2$ | $C1,P0$ | $C1,P1$ | $C1,P2$ | $C2,P0$ | $C2,P1$ | $C2,P2$ |
|---|---|---|---|---|---|---|---|---|---|
| $H=0$ | 0 | 0 | 0 | 0 | 0.5 | 1 | 0 | 1 | 0.5 |
| $H=1$ | 0.5 | 0 | 1 | 0 | 0 | 0 | 1 | 0 | 0.5 |
| $H=2$ | 0.5 | 1 | 0 | 1 | 0.5 | 0 | 0 | 0 | 0 |

Exemple de citire:

- coloana $(C=0,P=0)$: concurentul a ales ușa cu premiul, deci gazda deschide ușa $1$ sau ușa $2$, fiecare cu $0.5$;
- coloana $(C=0,P=1)$: ușa $0$ este aleasă, premiul e în spatele ușii $1$, deci gazda poate deschide doar ușa $2$ (probabilitate $1$);
- coloana $(C=0,P=2)$: gazda poate deschide doar ușa $1$;
- ușa deschisă de gazdă are întotdeauna probabilitate $0$ dacă este ușa aleasă sau ușa cu premiul (fiecare coloană are suma $1$).

**Asocierea CPD-urilor.** `model.add_cpds(cpd_c, cpd_p, cpd_h)` leagă tabelele de structura rețelei, iar `model.get_cpds()` returnează lista CPD-urilor asociate modelului.

**Validarea modelului.** `model.check_model()` verifică faptul că graful este aciclic, că fiecare variabilă are un CPD, că dimensiunile CPD-urilor corespund numărului de stări ale părinților și că probabilitățile din fiecare coloană însumează $1$. Returnează `True` dacă modelul este valid.

In [3]:
model.check_model()

True

**Inferența.** `VariableElimination(model)` creează un obiect de inferență exactă prin eliminarea variabilelor: variabilele care nu apar în interogare și nici în evidență sunt eliminate prin însumare. Metoda `query(["P"], evidence={"C": 0, "H": 2})` calculează distribuția a posteriori $P(P\mid C=0, H=2)$: concurentul a ales ușa $0$, iar gazda a deschis ușa $2$. Argumentul `show_progress=False` dezactivează bara de progres.

In [4]:
from pgmpy.inference import VariableElimination

infer = VariableElimination(model)
posterior_p = infer.query(["P"], evidence={"C": 0, "H": 2}, show_progress=False)
print(posterior_p)

+------+----------+
| P    |   phi(P) |
+======+==========+
| P(0) |   0.3333 |
+------+----------+
| P(1) |   0.6667 |
+------+----------+
| P(2) |   0.0000 |
+------+----------+


## Interpretarea rezultatului

Rezultatul se verifică cu regula lui Bayes. Pentru $p\in\{0,1,2\}$:

$$P(P=p\mid C=0,H=2)=\frac{P(H=2\mid C=0,P=p)\,P(P=p)}{\sum_{q}P(H=2\mid C=0,P=q)\,P(P=q)}$$

Numărătorii (cu $P(P=p)=\frac13$) sunt:

- $p=0$: $\frac12\cdot\frac13=\frac16$;
- $p=1$: $1\cdot\frac13=\frac13$;
- $p=2$: $0\cdot\frac13=0$.

Suma lor este $\frac16+\frac13=\frac12$, deci

$$P(P=0\mid C=0,H=2)=\frac{1/6}{1/2}=\frac13,\qquad P(P=1\mid C=0,H=2)=\frac{1/3}{1/2}=\frac23,\qquad P(P=2\mid C=0,H=2)=0$$

Ușa aleasă inițial ($0$) rămâne cu probabilitatea de $\frac13$ de a ascunde premiul, pentru că gazda ar fi deschis oricum o ușă cu capră. Probabilitatea $\frac23$ pe care o aveau împreună celelalte două uși se concentrează acum pe singura ușă rămasă închisă ($1$). Concluzie: **schimbarea ușii dublează șansa de câștig** (de la $\frac13$ la $\frac23$).

# Exercițiul 2 — Diagnostic medical cu rețea bayesiană

## a) Definirea modelului

Variabilele binare sunt $B$ (boala), $T$ (tuse), $D$ (dificultate de respirație) și $X$ (radiografie anormală). Din enunț rezultă structura $B\to T$, $B\to X$, $B\to D$, $T\to D$, iar distribuția comună se factorizează astfel:

$$P(B,T,X,D)=P(B)\,P(T\mid B)\,P(X\mid B)\,P(D\mid B,T)$$

CPD-ul lui $D$ are patru coloane, corespunzătoare combinațiilor $(B,T)$ în ordinea $(0,0),(0,1),(1,0),(1,1)$:

| | $B0,T0$ | $B0,T1$ | $B1,T0$ | $B1,T1$ |
|---|---|---|---|---|
| $D=0$ | 0.9 | 0.5 | 0.6 | 0.1 |
| $D=1$ | 0.1 | 0.5 | 0.4 | 0.9 |

In [5]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model2 = DiscreteBayesianNetwork([("B", "T"), ("B", "X"), ("B", "D"), ("T", "D")])

cpd_b = TabularCPD("B", 2, [[0.9], [0.1]])
cpd_t = TabularCPD("T", 2, [[0.7, 0.2], [0.3, 0.8]], evidence=["B"], evidence_card=[2])
cpd_x = TabularCPD("X", 2, [[0.9, 0.1], [0.1, 0.9]], evidence=["B"], evidence_card=[2])
cpd_d = TabularCPD(
    "D",
    2,
    [[0.9, 0.5, 0.6, 0.1], [0.1, 0.5, 0.4, 0.9]],
    evidence=["B", "T"],
    evidence_card=[2, 2],
)

model2.add_cpds(cpd_b, cpd_t, cpd_x, cpd_d)
model2.check_model()

True

## b) $P(B=1\mid T=1, D=1)$

In [6]:
infer2 = VariableElimination(model2)
print(infer2.query(["B"], evidence={"T": 1, "D": 1}, show_progress=False))

+------+----------+
| B    |   phi(B) |
+======+==========+
| B(0) |   0.6522 |
+------+----------+
| B(1) |   0.3478 |
+------+----------+


## c) Probabilitatea ca radiografia să fi fost anormală, știind că pacientul nu avea boala

In [7]:
print(infer2.query(["X"], evidence={"B": 0}, show_progress=False))
print(infer2.query(["X"], evidence={"B": 0, "T": 1, "D": 1}, show_progress=False))

+------+----------+
| X    |   phi(X) |
+======+==========+
| X(0) |   0.9000 |
+------+----------+
| X(1) |   0.1000 |
+------+----------+
+------+----------+
| X    |   phi(X) |
+======+==========+
| X(0) |   0.9000 |
+------+----------+
| X(1) |   0.1000 |
+------+----------+


## d) Calculul cu regula lui Bayes

**Punctul b).** Se sumează după $X$ în distribuția comună; deoarece $\sum_x P(X=x\mid B)=1$, rămâne

$$P(B,T,D)=P(B)\,P(T\mid B)\,P(D\mid B,T)$$

Atunci:

$$P(B=1\mid T=1,D=1)=\frac{P(B=1)\,P(T=1\mid B=1)\,P(D=1\mid B=1,T=1)}{\sum_{b\in\{0,1\}}P(B=b)\,P(T=1\mid B=b)\,P(D=1\mid B=b,T=1)}$$

Numărătorul:

$$0{,}1\cdot 0{,}8\cdot 0{,}9=0{,}072$$

Termenul pentru $B=0$ din suma de la numitor:

$$0{,}9\cdot 0{,}3\cdot 0{,}5=0{,}135$$

Prin urmare:

$$P(B=1\mid T=1,D=1)=\frac{0{,}072}{0{,}072+0{,}135}=\frac{0{,}072}{0{,}207}=\frac{8}{23}\approx 0{,}3478$$

**Punctul c).** În rețea, $X$ are ca unic părinte pe $B$. Atunci $X$ este independentă condiționat de $T$ și $D$ dacă se cunoaște $B$ ($B$ blochează toate drumurile dintre $X$ și $\{T,D\}$: $X\leftarrow B\rightarrow T$, $X\leftarrow B\rightarrow D$ și $X\leftarrow B\rightarrow T\rightarrow D$). Prin urmare, informația despre tuse și dificultatea de respirație nu mai contează odată ce se știe că pacientul nu are boala:

$$P(X=1\mid B=0,T=1,D=1)=P(X=1\mid B=0)=0{,}1$$

Probabilitatea ca radiografia să fi fost anormală este deci $0{,}1$, iar cea ca să fi fost normală este $0{,}9$.

# Exercițiul 3 — Jocul dintre $J_0$ și $J_1$

Notații: $p_0=\frac12$ (moneda lui $J_0$) și $p_1=\frac47$ (moneda măsluită a lui $J_1$).

## a) Simularea a 10000 de jocuri

In [8]:
import numpy as np

rng = np.random.default_rng(42)
p_moneda = {0: 0.5, 1: 4 / 7}
victorii = {0: 0, 1: 0}
n_jocuri = 10000

for _ in range(n_jocuri):
    primul = int(rng.integers(2))
    al_doilea = 1 - primul
    n = int(rng.integers(1, 7))
    m = int((rng.random(2 * n) < p_moneda[al_doilea]).sum())
    castigator = primul if n >= m else al_doilea
    victorii[castigator] += 1

for j in (0, 1):
    print(f"J{j}: {victorii[j]} victorii, probabilitate estimată = {victorii[j] / n_jocuri:.4f}")
print(f"Șanse mai mari are J{max(victorii, key=victorii.get)}")

J0: 4244 victorii, probabilitate estimată = 0.4244
J1: 5756 victorii, probabilitate estimată = 0.5756
Șanse mai mari are J1


## b) Rețeaua bayesiană

Variabilele modelului:

- $S\in\{0,1\}$ — jucătorul care începe ($S=0$: $J_0$ începe, $S=1$: $J_1$ începe), $P(S=0)=P(S=1)=\frac12$;
- $N\in\{1,\dots,6\}$ — rezultatul zarului din prima rundă, $P(N=n)=\frac16$, independent de $S$ (ambele zaruri sunt normale);
- $M\in\{0,\dots,12\}$ — numărul de steme din a doua rundă. Jucătorul din a doua rundă aruncă moneda de $2N$ ori, deci $M\mid S,N\sim\text{Binomial}(2N,\,p)$, cu $p=\frac47$ dacă $S=0$ (aruncă $J_1$) și $p=\frac12$ dacă $S=1$ (aruncă $J_0$);
- $W\in\{0,1\}$ — câștigătorul. Jucătorul din prima rundă câștigă dacă $N\ge M$, altfel câștigă cel din a doua rundă; $W$ este determinată de $(S,N,M)$.

Structura: $S\to M$, $N\to M$, $S\to W$, $N\to W$, $M\to W$.

In [9]:
from scipy.stats import binom
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

val_S = [0, 1]
val_N = [1, 2, 3, 4, 5, 6]
val_M = list(range(13))
val_W = [0, 1]
p_moneda = {0: 4 / 7, 1: 0.5}

model3 = DiscreteBayesianNetwork([("S", "M"), ("N", "M"), ("S", "W"), ("N", "W"), ("M", "W")])

cpd_s = TabularCPD("S", 2, [[0.5], [0.5]], state_names={"S": val_S})
cpd_n = TabularCPD("N", 6, [[1 / 6]] * 6, state_names={"N": val_N})

valori_m = [
    [binom.pmf(m, 2 * n, p_moneda[s]) if m <= 2 * n else 0.0 for s in val_S for n in val_N]
    for m in val_M
]
cpd_m = TabularCPD(
    "M",
    13,
    valori_m,
    evidence=["S", "N"],
    evidence_card=[2, 6],
    state_names={"M": val_M, "S": val_S, "N": val_N},
)

coloane_w = []
for s in val_S:
    for n in val_N:
        for m in val_M:
            primul = s
            castigator = primul if n >= m else 1 - primul
            coloane_w.append([1.0 if castigator == w else 0.0 for w in val_W])
valori_w = [[col[w] for col in coloane_w] for w in val_W]
cpd_w = TabularCPD(
    "W",
    2,
    valori_w,
    evidence=["S", "N", "M"],
    evidence_card=[2, 6, 13],
    state_names={"W": val_W, "S": val_S, "N": val_N, "M": val_M},
)

model3.add_cpds(cpd_s, cpd_n, cpd_m, cpd_w)
model3.check_model()

True

Verificare: probabilitățile de câștig calculate exact cu modelul, de comparat cu estimările din simulare.

In [10]:
infer3 = VariableElimination(model3)
print(infer3.query(["W"], show_progress=False))

+------+----------+
| W    |   phi(W) |
+======+==========+
| W(0) |   0.4267 |
+------+----------+
| W(1) |   0.5733 |
+------+----------+


## c) Cine a început jocul, știind că în a doua rundă s-a obținut o singură stemă

Se calculează $P(S\mid M=1)$.

In [11]:
posterior_s = infer3.query(["S"], evidence={"M": 1}, show_progress=False)
print(posterior_s)
print(f"Cel mai probabil a început J{posterior_s.state_names['S'][posterior_s.values.argmax()]}")

+------+----------+
| S    |   phi(S) |
+======+==========+
| S(0) |   0.4529 |
+------+----------+
| S(1) |   0.5471 |
+------+----------+
Cel mai probabil a început J1


**Concluzii.**

- a) Atât simularea, cât și calculul exact cu modelul arată că $J_1$ are șanse mai mari de câștig ($P(W=1)\approx 0{,}573$, față de $P(W=0)\approx 0{,}427$), deși moneda lui este măsluită în favoarea sa doar în a doua rundă.
- c) $P(S=1\mid M=1)\approx 0{,}547>P(S=0\mid M=1)\approx 0{,}453$, deci este mai probabil ca jocul să fi fost început de $J_1$.